<a href="https://colab.research.google.com/github/DeepthiManthapuram/Building_LLM_Applications/blob/main/RAG_DocuChat_Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LOAD DOCUMENT

In [1]:
!pip install -qU langchain-community pypdf

In [2]:
from langchain_community.document_loaders import PyPDFLoader
file_path = "https://arxiv.org/pdf/1706.03762"
loader = PyPDFLoader(file_path)
doc = loader.load()
print(doc)

/tmp/ipykernel_18923/766885366.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/1706.03762', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoo

In [3]:
#To read content in page
print(doc[0].page_content)

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exper

In [4]:
print(doc[2].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/1706.03762', 'total_pages': 15, 'page': 2, 'page_label': '3'}


SPLITTING DOCUMENT

In [5]:
!pip install -qU langchain-text-splitters

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

all_splits = text_splitter.split_documents(doc)
print(all_splits)
print(f"Paper split into {len(all_splits)} sub-documents")
print(f"metadata: {all_splits[0].metadata}")

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://arxiv.org/pdf/1706.03762', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoo

CREATING EMBEDDINGS

In [7]:
!pip install -qU langchain langchain-huggingface sentence_transformers

In [8]:
from langchain_core import embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model = "sentence-transformers/all-mpnet-base-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SAVE THE EMBEDDINGS TO VECTOR STORE

In [9]:
!pip install -qU chromadb
!pip install -qU opentelemetry-api opentelemetry-sdk

In [10]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name = "research_collection",
    embedding_function = embedding_model,
    persist_directory = "./chroma_lanchain_db"
)

document_ids = vector_store.add_documents(documents = all_splits)
print(document_ids)

['fe955884-8009-4e63-9b38-d402c9811db7', '0f8c9695-e427-4fb7-b2ef-8bfe75e83a07', '1c6017b3-59ab-4d7b-98ff-a3e6563a7039', '30e0653f-f69e-49f4-b0ba-64cdadf7cdb3', '748979d6-86ae-4959-8043-0b9feeedd804', '852703e7-73fb-4cf1-904f-5a550bc05f92', 'f15aaa4f-9b84-43e0-ab78-e467cae374c4', 'd01bec97-4f5b-408a-8ba7-85e6b74fa53b', 'cfe2e5f3-bc51-4b58-815b-f5d4590d005a', '29dfbf98-279a-47b2-865e-f76bc3a5d461', '1e88a899-5a87-4557-aebd-5e1ffe73d1fb', 'c64b79c5-8034-4ad7-ba51-c5147e6c6b01', '7b5a6fd9-3bfd-4590-9b27-9f38f410a29f', 'bd89252c-31c1-42c5-8677-ee1bac6562b2', '460e720d-0ed0-4631-8ec2-581d5180e03b', 'd5b7396f-c00c-4e78-a8f6-ee05cf076228', '44c7e4a6-da6f-4531-87a9-e7ecbdd47191', 'e740297d-74ca-4eb1-a454-41aa00f0e6f9', '0337f6d8-8fd1-454d-8e22-ffa4240dacd0', 'd0423960-39e6-4a40-8585-05aaae108a80', 'b7bc18b7-e29f-4d8d-a970-fcc435a8a470', '437e7b29-0dba-481e-a157-4d749426ae2d', '95b40688-5c38-4608-83a1-1825366dc60f', '42819ba7-038d-4a87-aaf3-bcbc6b622b5a', '0e142cdb-860c-424a-a0bf-f7f4b7c4c10b',

In [11]:
sample = vector_store.get(limit = 1, include = ["embeddings", "documents"])
print(sample)

{'ids': ['326047a1-93ea-49d4-a5a3-ce8eebdf2bf3'], 'embeddings': array([[ 3.45199718e-03,  1.59770716e-02, -1.30287064e-02,
         9.54000861e-04, -5.11657149e-02, -1.46142941e-03,
         3.13304574e-03, -2.18834542e-02, -6.23742379e-02,
        -3.53030534e-03,  2.20327987e-03, -3.70261446e-02,
         2.37597041e-02,  4.24362980e-02,  5.18797114e-02,
        -1.31552778e-02,  4.10710797e-02,  1.13152806e-02,
        -3.28718573e-02,  2.76140347e-02, -3.53557654e-02,
        -3.68874781e-02,  2.41961647e-02,  4.08427790e-02,
         3.06257885e-02,  8.32777563e-03,  7.36685935e-04,
        -3.48431394e-02,  3.06146462e-02, -3.79032455e-02,
         9.24858823e-03,  3.78236063e-02,  2.70037800e-02,
         6.65727481e-02,  2.15587761e-06, -1.03399744e-02,
         2.15955116e-02, -3.54087278e-02,  1.28482422e-02,
         3.28594673e-04, -3.07061672e-02, -3.66120487e-02,
        -1.32616656e-02, -8.63518938e-03, -2.98039783e-02,
        -4.66191396e-03,  7.08968192e-02,  5.038664

In [12]:
print(document_ids[:3])

['fe955884-8009-4e63-9b38-d402c9811db7', '0f8c9695-e427-4fb7-b2ef-8bfe75e83a07', '1c6017b3-59ab-4d7b-98ff-a3e6563a7039']


RETRIEVE AND GENERATE

In [13]:
def retrieve_context(query: str, k:int = 2):
  retrieved_docs = vector_store.similarity_search(query, 2)

  docs_content = ""
  for doc in retrieved_docs:
    docs_content += f"source:{doc.metadata}\n"
    docs_content += f"content:{doc.page_content}\n"

  return docs_content, retrieved_docs

GENERATE ANSWERS USING LLM

In [14]:
!pip install -U langchain-google-genai

In [15]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
model = init_chat_model(
    "google_genai:gemini-3.6-flash",
    api_key = api_key
)

In [17]:
def docu_chat(user_query):
  context, source_docs = retrieve_context(user_query, k=2)
  system_message = f"""You are a helpful chatbot.
                    Use only the following pieces of context to answer the
                    question. Don't makeup any new information: {context}"""
  messages = [
      {"role":"system", "content":system_message},
      {"role":"user", "content":user_query},
  ]
  response = model.invoke(messages)
  return{
      "answer": response.content,
      "source_documents": source_docs,
      "context_used": context
  }

result = docu_chat("Explain decoders in transformers?")
print(result)
print(result["answer"])

{'answer': [{'type': 'text', 'text': 'Based on the provided context, the decoder in the Transformer uses multi-head attention in two primary ways:\n\n1. **Encoder-Decoder Attention Layers:** \n   * **Queries** come from the previous decoder layer.\n   * **Memory keys and values** come from the output of the encoder.\n   * This setup allows every position in the decoder to attend over all positions in the input sequence.\n\n2. **Decoder Self-Attention Layers:**\n   * These layers allow each position in the decoder to attend to all positions in the decoder up to and including that position (with mechanisms in place to prevent leftward information flow).', 'extras': {'signature': 'ErkUCrYUAWkUfRMiZhtH3glADzxfdXkasp99ANQ7gHQr1WvuwhtzBnN987p91EhhemPwsg3OgUJG3Bii07RYAD5NZp95K3hyBY9Jj3O3HBY+xhp8Qfta2RESzkKQOJG8jNESyw+AJdVSAJNb+D0KuUZpI3WDnvoyUyB/MwgujrLO2qvnVH5/DuLdN6i4PkUEZhNX66kBVj6Ub1XJaGQ13NnvqrY3W9hJrj5vU+ocaaLrUffOn9PDEAp5P9b3W+55AvM05UPGMsU6GlcwO6NgbElLaiJvTfVUyPMb/1lsE426fLf0qZHeEPh00